In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd


In [6]:
def create_webdriver():
    opts = Options()
    # opts.add_argument("--headless=new")  # enable for no-GUI scraping
    # opts.add_experimental_option("detach", True)  # keep window open after script ends
    return webdriver.Chrome(options=opts)


WEBSITE = "https://old.reddit.com/r/wallstreetbets/"
# https://old.reddit.com/r/wallstreetbets/
# https://books.toscrape.com/
driver = create_webdriver()
driver.get(WEBSITE)


In [7]:
# Wait until the project links are present
titles = []
post_idList = []
created_utcList = []
flairsList = []
# upvotesList = []
num_commentsList = []
permalinkList = []

i = 0

while i < 3:
    wait = WebDriverWait(driver, 10)
    things = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.thing"))
    )




    for thing in things:
        if "stickied" in thing.get_attribute("class"):
            continue
        if thing.get_attribute("data-promoted") == "true":
            continue
        title = thing.find_element(By.CSS_SELECTOR, 'p.title > a')
        titles.append(title.text)

        post_id = thing.get_attribute("data-fullname")
        post_idList.append(post_id)

        time_elements = thing.find_elements(By.TAG_NAME, "time")
        if time_elements:
            created_utc = time_elements[0].get_attribute("datetime")
        else:
            created_utc = None
        created_utcList.append(created_utc)

        flairs = thing.find_element(By.CSS_SELECTOR, 'span.linkflairlabel')
        flairsList.append(flairs.text)

        # upvotes = thing.find_element(By.CSS_SELECTOR, 'div.score')
        # upvotesList.append(upvotes.text)

        comments = thing.find_element(By.CSS_SELECTOR, 'a.comments')
        num_commentsList.append(comments.text)

        permalink = comments.get_attribute("href")
        permalinkList.append(permalink)

        


    
    next_button = WebDriverWait(driver, 10).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'span.next-button')))
    if next_button.get_attribute('Disabled'):
        break  # Exit the loop if the next button is disabled
    else:
        # Click the next button to navigate to the next page
        next_button.click()
    i += 1


driver.quit()


In [8]:
df = pd.DataFrame({'titles': titles, 'post_idList': post_idList, 'created_utc': created_utcList, 'flairs': flairsList, 'num_comments': num_commentsList, 'permalink': permalinkList})
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 500)  # adjust for your screen
pd.set_option('display.max_columns', None)
print(df)

                                                                                                                                          titles post_idList                created_utc              flairs    num_comments                                                                                                    permalink
0                                                                      My biggest trade yet, doesn’t feel real to be doing this at 22 years old.  t3_1mmz3e7  2025-08-11T00:44:37+00:00                Gain    215 comments  https://old.reddit.com/r/wallstreetbets/comments/1mmz3e7/my_biggest_trade_yet_doesnt_feel_real_to_be_doing/
1                                                                                             NVDA and AMD hit with 15% Tariff on China Revenues  t3_1mmuggu  2025-08-10T21:18:51+00:00                News    246 comments  https://old.reddit.com/r/wallstreetbets/comments/1mmuggu/nvda_and_amd_hit_with_15_tariff_on_china_revenues/
2            

# Next Steps:
- data cleaning
- perform analytics
- visualizations

In [ ]:
df['titles'] = df['titles'].str.replace('\n', ' ', regex=False).str.strip()
df['flairs'] = df['flairs'].fillna('').str.strip()
df = df[df['flair'] != 'Daily Discussion']

In [ ]:
df['num_comments'] = (df['num_comments'].str.extract('(\d+)').fillna(0).astype(int))
df['num_comments']

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\jlee0\AppData\Local\Temp\ipykernel_9916\1112965993.py:1: SyntaxWarning: invalid escape sequence '\d'
  df['num_comments'] = (df['num_comments'].str.extract('(\d+)').fillna(0).astype(int))


0       215
1       246
2       132
3        54
4        15
5        20
6       256
7       502
8        11
9         9
10       91
11      153
12      305
13       21
14       95
15       85
16      328
17     1057
18       28
19       21
20      308
21      392
22      193
23       70
24      523
25       46
26      125
27      135
28      246
29       28
30       83
31    10147
32       25
33       85
34       51
35       37
36       59
37      378
38       11
39      111
40      133
41       28
42       73
43       21
44       18
45       10
46       58
47    10839
48       11
49       19
50       19
51      105
52       58
53       42
54       29
55       17
56      965
57        8
58        3
59       26
60       12
61       12
62       16
63       10
64       96
65        5
66       81
67       21
68      147
69      528
70        4
71       90
72       66
73        1
74      150
Name: num_comments, dtype: int64

In [14]:
df['created_utc'] = pd.to_datetime(df['created_utc'], utc=True)
df['date'] = df['created_utc'].dt.date

In [18]:
pattern = r'\b[A-Z]{1,5}\b'
df['tickers'] = df['titles'].str.findall(pattern)
df['tickers']


0                                                   []
1                                          [NVDA, AMD]
2                                                  [I]
3                                               [NVDA]
4                                        [COIN, PRINT]
5                                         [YOLO, MSOX]
6                                          [UNH, YOLO]
7                                        [WE, DID, IT]
8                                             [UNH, I]
9                                              [STOCK]
10                                                  []
11                                               [YTD]
12                                          [AI, S, P]
13                                         [HOOD, AMD]
14                                                 [I]
15                                                  []
16                                              [I, I]
17                                                  []
18        